# NB-2: HotPotQA + MuSiQue Multi-Hop Benchmarks
**Publication blocker PB-13 (L3 hop isolation)**

Runs two multi-hop QA benchmarks:
- **HotPotQA** — 2-hop retrieval over Wikipedia-style passages
- **MuSiQue** — multi-hop reasoning (run twice: with L3 and without L3)

Comparing MuSiQue with/without L3 isolates the knowledge-graph hop contribution.

**Datasets:** hotpotqa_dev.json (7405 Q, we use 50), musique_dev.jsonl (200 Q)

**Time estimate:** ~15 min per run

## Step 1 — Install & clone

In [ ]:
import os, subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'sentence-transformers', 'hnswlib', 'python-dotenv', 'groq', 'requests'], check=True)

REPO = 'https://github.com/Lamaq-Mujpurwala/CSAM-IPD-HALH.git'
REPO_DIR = '/kaggle/working/CSAM-IPD-HALH' if os.path.exists('/kaggle') else '/content/CSAM-IPD-HALH'

if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO, REPO_DIR], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'csam_project'))
print(f'Ready in {os.getcwd()}')

## Step 2 — API key
**Kaggle:** Notebook → Settings → Secrets → `GROQ_API_KEY`

**Colab:** Left sidebar key icon → `GROQ_API_KEY`

In [ ]:
import os

def load_api_key():
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('GROQ_API_KEY')
        if key: os.environ['GROQ_API_KEY'] = key; return 'kaggle'
    except Exception: pass
    try:
        from google.colab import userdata
        key = userdata.get('GROQ_API_KEY')
        if key: os.environ['GROQ_API_KEY'] = key; return 'colab'
    except Exception: pass
    if os.environ.get('GROQ_API_KEY'): return 'env'
    raise RuntimeError('Add GROQ_API_KEY to Kaggle/Colab Secrets')

print(f'API key loaded from: {load_api_key()}')
with open('.env', 'w') as f:
    f.write(f"GROQ_API_KEY={os.environ['GROQ_API_KEY']}\n")

## Step 3 — Configure

In [ ]:
PROVIDER   = 'groq'
MODEL      = 'llama-3.1-8b-instant'
SEED       = 42
CHECKPOINT = '/kaggle/working' if os.path.exists('/kaggle') else '/content'

# HotPotQA: how many questions (max 7405, use 50 for quick, 200 for paper)
HOTPOT_Q   = 50
HOTPOT_DS  = 'csam_project/benchmarks/data/hotpotqa_dev.json'

# MuSiQue: uses all 200 questions in the dev set
MUSIQUE_DS = 'csam_project/benchmarks/data/musique_dev.jsonl'

safe_model = MODEL.replace('/', '_')
OUT_HOTPOT       = f'csam_project/benchmarks/results_hotpotqa_{PROVIDER}_{safe_model}.json'
OUT_MUSIQUE      = f'csam_project/benchmarks/results_musique_{PROVIDER}_{safe_model}.json'
OUT_MUSIQUE_NOL3 = f'csam_project/benchmarks/results_musique_{PROVIDER}_{safe_model}_nol3.json'

print(f'Model: {MODEL} | HotPotQA Q: {HOTPOT_Q} | MuSiQue: all 200')

## Step 4 — Run HotPotQA

In [ ]:
import subprocess, sys, os
os.chdir(REPO_DIR)

cmd = [
    sys.executable, '-m', 'csam_project.benchmarks.benchmark_hotpotqa',
    '--provider', PROVIDER,
    '--model', MODEL,
    '--questions', str(HOTPOT_Q),
    '--dataset', HOTPOT_DS,
    '--seed', str(SEED),
    '--checkpoint-dir', CHECKPOINT,
]
print('Running HotPotQA...')
result = subprocess.run(cmd, capture_output=False, text=True)
print('HotPotQA done' if result.returncode == 0 else 'HotPotQA FAILED')

## Step 5 — Run MuSiQue (with L3)
Full CSAM system — L3 knowledge graph enables multi-hop retrieval.

In [ ]:
cmd = [
    sys.executable, '-m', 'csam_project.benchmarks.benchmark_musique',
    '--provider', PROVIDER,
    '--model', MODEL,
    '--seed', str(SEED),
    '--checkpoint-dir', CHECKPOINT,
]
print('Running MuSiQue WITH L3...')
result = subprocess.run(cmd, capture_output=False, text=True)
print('MuSiQue+L3 done' if result.returncode == 0 else 'MuSiQue+L3 FAILED')

## Step 6 — Run MuSiQue (WITHOUT L3)
Ablation: same setup but L3 knowledge graph is disabled.
The F1 delta vs the L3 run quantifies the graph hop contribution (PB-13).

In [ ]:
cmd = [
    sys.executable, '-m', 'csam_project.benchmarks.benchmark_musique',
    '--provider', PROVIDER,
    '--model', MODEL,
    '--seed', str(SEED),
    '--checkpoint-dir', CHECKPOINT,
    '--no-l3',   # ← disables L3 knowledge graph
]
print('Running MuSiQue WITHOUT L3...')
result = subprocess.run(cmd, capture_output=False, text=True)
print('MuSiQue-noL3 done' if result.returncode == 0 else 'MuSiQue-noL3 FAILED')

## Step 7 — Results summary

In [ ]:
import json, os

print('=' * 60)
print('RESULTS SUMMARY')
print('=' * 60)

def show(label, path):
    if not os.path.exists(path):
        print(f'{label:<30} NOT FOUND ({path})')
        return
    with open(path) as f:
        d = json.load(f)
    f1 = d.get('micro_f1') or d.get('avg_f1') or d.get('overall_f1', 0)
    n  = d.get('num_questions') or len(d.get('per_conversation', []))
    print(f'{label:<30} F1={f1:.4f}  n={n}')

show('HotPotQA (CSAM)',   OUT_HOTPOT)
show('MuSiQue (with L3)', OUT_MUSIQUE)
show('MuSiQue (no L3)',   OUT_MUSIQUE_NOL3)

# L3 hop delta
if os.path.exists(OUT_MUSIQUE) and os.path.exists(OUT_MUSIQUE_NOL3):
    with open(OUT_MUSIQUE) as f: m_l3 = json.load(f)
    with open(OUT_MUSIQUE_NOL3) as f: m_nl3 = json.load(f)
    f1_l3  = m_l3.get('micro_f1') or m_l3.get('avg_f1', 0)
    f1_nl3 = m_nl3.get('micro_f1') or m_nl3.get('avg_f1', 0)
    print(f'\nL3 hop contribution: {f1_l3 - f1_nl3:+.4f} F1 points')

## Step 8 — Save output files

In [ ]:
import shutil

files_to_save = [OUT_HOTPOT, OUT_MUSIQUE, OUT_MUSIQUE_NOL3]

if os.path.exists('/kaggle'):
    for fp in files_to_save:
        if os.path.exists(fp):
            dest = os.path.join('/kaggle/working', os.path.basename(fp))
            shutil.copy(fp, dest)
            print(f'Kaggle output: {dest}')
else:
    try:
        from google.colab import files
        for fp in files_to_save:
            if os.path.exists(fp): files.download(fp); print(f'Downloaded: {fp}')
    except ImportError:
        print('Files at:', [fp for fp in files_to_save if os.path.exists(fp)])